In [1]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# Column coverage

In [2]:
parquet_path = Path("Geo/merged_all_geo.parquet")
output_path = Path("Final/column_coverage.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(parquet_path)

column_coverage = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "non_null_count": [df[col].notna().sum() for col in df.columns],
    "total_count": len(df),
})

column_coverage["coverage_pct"] = (
    column_coverage["non_null_count"] / column_coverage["total_count"] * 100
)

column_coverage.to_csv(output_path, index=False)
print(len(df))
print(f"Saved: {output_path}")
print(column_coverage)

3736407
Saved: Final\column_coverage.csv
                      column    dtype  non_null_count  total_count  \
0     acceptedScientificName   string         3736394      3736407   
1           acceptedTaxonKey    Int64         2400481      3736407   
2                    aphiaID    Int64         3736407      3736407   
3            acceptedAphiaID    Int64         3736407      3736407   
4                areaSampled  float64          342014      3736407   
..                       ...      ...             ...          ...   
63                      year    Int64         3650633      3736407   
64                      Keep      str         3288475      3736407   
65                  isMarine  boolean         3735012      3736407   
66                isBrackish  boolean         3005417      3736407   
67  geomorphologicalCategory      str         3736407      3736407   

    coverage_pct  
0      99.999652  
1      64.245704  
2     100.000000  
3     100.000000  
4       9.153553  
..  

# Fixes

## Remove redundant & empty columns

In [3]:
parquet_path = Path("Geo/merged_all_geo.parquet")
df = pd.read_parquet(parquet_path)

# Remove specified columns and save to Final directory
columns_to_remove = ["trawlingDistance", "depthA", "depthB", "Keep"]
final_df = df.drop(columns=[col for col in columns_to_remove if col in df.columns])

final_path = Path("Final/merged_all_final.parquet")
final_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(final_df.shape)

Saved: Final\merged_all_final.parquet
(3736407, 67)


## Fill in flagged column

In [4]:
# Update the 'flagged' column: set to True where 'flaggedComment' is not null
final_df['flagged'] = final_df['flagged'] | final_df['flaggedComment'].notna()

# Save the updated dataframe
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")

Saved: Final\merged_all_final.parquet


## Check for mojibake issues

In [5]:
# Load the final parquet and check for mojibake issues
final_df = pd.read_parquet(final_path)

mojibake_patterns = [
    "�",      # replacement character
    "â€",     # common UTF-8/Windows-1252 mojibake prefix
    "â€™",
    "â€œ",
    "â€�",
    "â€ž",
    "ï¿½",     # replacement char rendered as mojibake
    "Ã",
    "Â",
]

# Check string columns for mojibake indicators
string_cols = final_df.select_dtypes(include=['string', 'object']).columns
mojibake_found = False

for col in string_cols:
    series = final_df[col].astype('string[python]')
    
    mojibake_mask = pd.Series(False, index=series.index)
    for pattern in mojibake_patterns:
        mojibake_mask = mojibake_mask | series.str.contains(pattern, na=False, regex=False)
    
    if mojibake_mask.any():
        mojibake_found = True
        print(f"\n--- Mojibake detected in '{col}' ({mojibake_mask.sum()} rows) ---")
        sample = final_df.loc[mojibake_mask, col].drop_duplicates().head(10)
        print(sample.values)

if not mojibake_found:
    print("No mojibake issues detected in Final/merged_all_final.parquet")


--- Mojibake detected in 'eventRemarks' (1 rows) ---
<ArrowStringArray>
['RV Akademik M.A. Lavrentyev _cruise 82_st.BS18 âˆšÃ¨17 T1']
Length: 1, dtype: string

--- Mojibake detected in 'verbatimID' (2 rows) ---
<ArrowStringArray>
['1764601749653-eu09oh2xu|185Ã', '1764601749653-eu09oh2xu|185Â']
Length: 2, dtype: string

--- Mojibake detected in 'verbatimScientificName' (200 rows) ---
<ArrowStringArray>
[                            'BonelliidaeÂ',
       'Micronephthys minuta (Th�el, 1879)',
     'Phyllodoce groenlandica �rsted, 1842',
   'Pectinaria granulata (Linnaeus, 1767)�',
              'Pholoe baltica �rsted, 1843',
         'Nephtys longosetosa �rsted, 1842',
 'Cirratulus cirratus (O. F. M�ller, 1776)',
       'Cylichna gouldii (Couthouy, 1839)�',
   'Scoletoma fragilis (O.F. M�ller, 1776)',
         'Nicolea zostericola �rsted, 1844']
Length: 10, dtype: string


In [6]:
# Check for mojibake (encoding issues) in string columns and repair recoverable text
mojibake_patterns = [
    "�",      # replacement character
    "â€",     # common UTF-8/Windows-1252 mojibake prefix
    "â€™",
    "â€œ",
    "â€�",
    "â€ž",
    "ï¿½",     # replacement char rendered as mojibake
    "Ã",
    "Â",
]

# Try to recover strings that were decoded with the wrong encoding
def fix_mojibake_text(value):
    if pd.isna(value) or not isinstance(value, str):
        return value

    for source_encoding in ("cp1252", "latin1"):
        try:
            fixed = value.encode(source_encoding).decode("utf-8")
            if fixed != value:
                return fixed
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass

    return value

# Check string columns for common mojibake indicators
string_cols = final_df.select_dtypes(include=['string', 'object']).columns
repaired_df = final_df.copy()
unrecoverable_examples = []

for col in string_cols:
    # Use Python-backed strings to avoid Arrow regex limitations
    series = repaired_df[col].astype('string[python]')

    mojibake_mask = pd.Series(False, index=series.index)
    for pattern in mojibake_patterns:
        mojibake_mask = mojibake_mask | series.str.contains(pattern, na=False, regex=False)

    if mojibake_mask.any():
        sample_original = repaired_df.loc[mojibake_mask, col].drop_duplicates().head(10)
        sample_corrected = sample_original.map(fix_mojibake_text)
        preview_df = pd.DataFrame({
            "original": sample_original.values,
            "corrected": sample_corrected.values,
        })
        preview_df = preview_df[preview_df["original"] != preview_df["corrected"]]

        if not preview_df.empty:
            print(f"\n--- Potential mojibake in '{col}' ({mojibake_mask.sum()} rows) ---")
            print(preview_df.to_string(index=False))

        unrecoverable_mask = sample_original == sample_corrected
        if unrecoverable_mask.any():
            unrecoverable_df = pd.DataFrame({
                "column": col,
                "value": sample_original[unrecoverable_mask].values,
            })
            unrecoverable_examples.append(unrecoverable_df)

    repaired_df[col] = repaired_df[col].map(fix_mojibake_text)

repaired_output_path = Path("Final/merged_all_final.parquet")
repaired_output_path.parent.mkdir(parents=True, exist_ok=True)
repaired_df.to_parquet(repaired_output_path, index=False)
print(f"\nSaved repaired parquet: {repaired_output_path}")

if unrecoverable_examples:
    unrecoverable_df = pd.concat(unrecoverable_examples, ignore_index=True).drop_duplicates()
    print("\n--- Potential entries not recovered by the repair pass ---")
    print(unrecoverable_df.head(20).to_string(index=False))
else:
    print("\nNo unrecoverable mojibake-like entries were found in the sampled values.")

print("\nMojibake repair complete!")


--- Potential mojibake in 'eventRemarks' (1 rows) ---
                                                 original                                              corrected
RV Akademik M.A. Lavrentyev _cruise 82_st.BS18 âˆšÃ¨17 T1 RV Akademik M.A. Lavrentyev _cruise 82_st.BS18 √è17 T1

Saved repaired parquet: Final\merged_all_final.parquet

--- Potential entries not recovered by the repair pass ---
                column                                    value
            verbatimID             1764601749653-eu09oh2xu|185Ã
            verbatimID             1764601749653-eu09oh2xu|185Â
verbatimScientificName                             BonelliidaeÂ
verbatimScientificName       Micronephthys minuta (Th�el, 1879)
verbatimScientificName     Phyllodoce groenlandica �rsted, 1842
verbatimScientificName   Pectinaria granulata (Linnaeus, 1767)�
verbatimScientificName              Pholoe baltica �rsted, 1843
verbatimScientificName         Nephtys longosetosa �rsted, 1842
verbatimScientificName Cirr

## Remove -999 values from individualCount

In [7]:
# Load the final parquet and replace -999 values with NaN in individualCount
final_df = pd.read_parquet(final_path)

# Replace -999 with NaN in individualCount column
final_df['individualCount'] = final_df['individualCount'].replace(-999.0, pd.NA)

# Save the updated dataframe
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(f"Replaced -999 values with NaN in individualCount")

Saved: Final\merged_all_final.parquet
Replaced -999 values with NaN in individualCount


## Check for other sentinel values that should be replaced with NaN

In [8]:
# Scan for common sentinel values that may need to be converted to NA/NaN
sentinel_values = [-999, -999.0, -9999, -9999.0, 999, 999.0, 9999, 9999.0, 99999, 99999.0]
sentinel_strings = {"-999", "-999.0", "-9999", "-9999.0", "999", "999.0", "9999", "9999.0", "99999", "99999.0"}

summary = []

for col in final_df.columns:
    s = final_df[col]

    if pd.api.types.is_numeric_dtype(s):
        for sentinel in sentinel_values:
            mask = s.eq(sentinel)
            if mask.any():
                summary.append({
                    "column": col,
                    "sentinel": sentinel,
                    "count": int(mask.sum()),
                    "sample": s.loc[mask].head(3).tolist(),
                })

    elif pd.api.types.is_string_dtype(s) or s.dtype == "object":
        ss = s.astype("string[python]")
        for sentinel in sentinel_strings:
            mask = ss.eq(sentinel)
            if mask.any():
                summary.append({
                    "column": col,
                    "sentinel": sentinel,
                    "count": int(mask.sum()),
                    "sample": ss.loc[mask].head(3).tolist(),
                })

summary_df = pd.DataFrame(summary).sort_values(["count", "column"], ascending=[False, True])

if summary_df.empty:
    print("No common sentinel values found.")
else:
    print("Common sentinel values found:")
    print(summary_df.to_string(index=False))

    # Optional: show columns with any sentinel hits
    print("\nColumns with sentinel hits:")
    print(summary_df["column"].drop_duplicates().tolist())

Common sentinel values found:
    column sentinel  count                   sample
gebcoDepth     -999     44 [-999.0, -999.0, -999.0]
gebcoDepth   -999.0     44 [-999.0, -999.0, -999.0]
  depthA_m      999     28    [999.0, 999.0, 999.0]
  depthA_m    999.0     28    [999.0, 999.0, 999.0]
  depthB_m      999     16    [999.0, 999.0, 999.0]
  depthB_m    999.0     16    [999.0, 999.0, 999.0]
   eventID      999     15          [999, 999, 999]
   eventID     9999      3       [9999, 9999, 9999]
verbatimID      999      2               [999, 999]
  sourceID    99999      1                  [99999]

Columns with sentinel hits:
['gebcoDepth', 'depthA_m', 'depthB_m', 'eventID', 'verbatimID', 'sourceID']


## Fill in missing acceptedScientificName

In [9]:
mask = final_df["acceptedScientificName"].isna() & final_df["acceptedAphiaID"].notna()

unique_accepted_aphia_ids = (
    final_df.loc[mask, "acceptedAphiaID"]
    .drop_duplicates()
    .sort_values()
)

print(unique_accepted_aphia_ids.to_string(index=False))

 128497
 182832
 182904
 206284
 741024
1584794


In [10]:
# Fill missing acceptedScientificName and validAuthority from acceptedAphiaID

aphia_map = {
    101070: {"acceptedScientificName": "Echinoderidae", "validAuthority": "Carus, 1885"},
    128497: {"acceptedScientificName": "Distichoptilum", "validAuthority": "Verrill, 1882"},
    182832: {"acceptedScientificName": "Nellobia", "validAuthority": "Fisher, 1946"},
    182904: {"acceptedScientificName": "Vitjazema", "validAuthority": "Zenkevitch, 1958"},
    206284: {"acceptedScientificName": "Zoanthus", "validAuthority": "Lamarck, 1801"},
    741024: {"acceptedScientificName": "Antheidae", "validAuthority": "Gosse, 1860"},
    1584794: {"acceptedScientificName": "Epomidiostomum", "validAuthority": "Skryabin, 1915"},
}

final_df = pd.read_parquet(final_path)

for aphia_id, values in aphia_map.items():
    mask = final_df["acceptedAphiaID"].eq(aphia_id)
    if "acceptedScientificName" in final_df.columns:
        final_df.loc[mask & final_df["acceptedScientificName"].isna(), "acceptedScientificName"] = values["acceptedScientificName"]
    if "validAuthority" in final_df.columns:
        final_df.loc[mask & final_df["validAuthority"].isna(), "validAuthority"] = values["validAuthority"]

final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")

Saved: Final\merged_all_final.parquet


## Fill in missing entries in OGSL records

In [11]:
# Load the final parquet and fill in missing OGSL dataset information
final_path = Path("Final/merged_all_final.parquet")
final_df = pd.read_parquet(final_path)

# Fill in datasetKey and datasetTitle for OGSL records
ogsl_mask = final_df["sourceDatabase"] == "OGSL"

final_df.loc[ogsl_mask, "datasetKey"] = "AmdBenthos"
final_df.loc[ogsl_mask, "datasetTitle"] = "Ecological Characterization of Benthos and Physicochemical Properties of Sediments (Granulometry, Organic Matter, Pigments) from the Amundsen Missions (2007-2024)"

# Save the updated dataframe
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(f"Filled in {ogsl_mask.sum()} OGSL records with dataset information")

Saved: Final\merged_all_final.parquet
Filled in 7236 OGSL records with dataset information


# Get number of taxa per station

In [12]:
stations = pd.read_parquet("Final/merged_all_final.parquet")

cols = [
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "datasetKey",
    "datasetTitle",
    "sourceDatabase",
    "year",
    "gebcoDepth",
    "startDayOfYear",
    "gear", 
]
all_stations = stations[cols].drop_duplicates()

total_stations = len(all_stations)
print(f"Total unique stations: {total_stations}")

all_stations.to_parquet("Final/all_stations.parquet", index=False)

all_stations

Total unique stations: 233516


,decimalLatitudeStart,decimalLongitudeStart,datasetKey,datasetTitle,sourceDatabase,year,gebcoDepth,startDayOfYear,gear
0,70.001883,-126.254916,83eac1c3-e549-4eba-9dad-01d81ba8b94c,DFO Arctic Region Biodiversity of the Benthic ...,OBIS,2022,-212.0,230,NaN
1,63.718060,-68.518610,8dbd0669-92d1-41ef-a5db-527bc6aa1ec9,Biodiversity of benthic assemblages on the Arc...,OBIS,1969,4.0,246,NaN
2,53.292600,-169.778670,f5a4799e-dc24-4807-89d9-01da47d52e3b,"NOAA, Deep-Sea Coral Research and Technology P...",OBIS,2023,-792.0,204,NaN
3,55.740002,-162.823334,89e23fc8-3f61-4480-9de3-358fe6eefe0b,National Museum of Natural History Invertebrat...,OBIS,1982,-57.0,304,NaN
4,72.351167,-119.073333,55ff4bf2-b3d2-49bf-8749-7c41fb2c0c19,DFO Arctic Region Biodiversity of the Benthic ...,OBIS,2024,-77.0,238,NaN
...,...,...,...,...,...,...,...,...,...
3734287,80.450000,-128.980000,Benthos ArcOD Sirenko (a),Occurrences of macro- and megabenthic organism...,CRITTERBASE,1977,-3510.0,193,Sigsbi trawl generic
3734288,80.130000,-127.800000,Benthos ArcOD Sirenko (a),Occurrences of macro- and megabenthic organism...,CRITTERBASE,1977,-2279.0,199,Sigsbi trawl generic
3734289,79.430000,-126.370000,Benthos ArcOD Sirenko (a),Occurrences of macro- and megabenthic organism...,CRITTERBASE,1977,-2515.0,221,Sigsbi trawl generic
3734290,79.370000,-127.700000,Benthos ArcOD Sirenko (a),Occurrences of macro- and megabenthic organism...,CRITTERBASE,1977,-3395.0,240,grab generic


In [13]:
# Count unique aphiaID per station row and overwrite Final/all_stations.parquet

count_col = "uniqueAphiaIDCount"

aphia_counts = (
    stations.groupby(cols, dropna=False)["acceptedAphiaID"]
    .nunique()
    .reset_index(name=count_col)
)

if count_col in all_stations.columns:
    all_stations = all_stations.drop(columns=[count_col])

all_stations = all_stations.merge(aphia_counts, on=cols, how="left")
all_stations[count_col] = all_stations[count_col].fillna(0).astype("Int64")

all_stations.to_parquet("Final/all_stations.parquet", index=False)

print(f"Saved: Final/all_stations.parquet")
print(all_stations[[*cols, count_col]].head())

Saved: Final/all_stations.parquet
   decimalLatitudeStart  decimalLongitudeStart  \
0             70.001883            -126.254916   
1             63.718060             -68.518610   
2             53.292600            -169.778670   
3             55.740002            -162.823334   
4             72.351167            -119.073333   

                             datasetKey  \
0  83eac1c3-e549-4eba-9dad-01d81ba8b94c   
1  8dbd0669-92d1-41ef-a5db-527bc6aa1ec9   
2  f5a4799e-dc24-4807-89d9-01da47d52e3b   
3  89e23fc8-3f61-4480-9de3-358fe6eefe0b   
4  55ff4bf2-b3d2-49bf-8749-7c41fb2c0c19   

                                        datasetTitle sourceDatabase  year  \
0  DFO Arctic Region Biodiversity of the Benthic ...           OBIS  2022   
1  Biodiversity of benthic assemblages on the Arc...           OBIS  1969   
2  NOAA, Deep-Sea Coral Research and Technology P...           OBIS  2023   
3  National Museum of Natural History Invertebrat...           OBIS  1982   
4  DFO Arctic Region 

# Datasets

In [14]:
datasets_path = Path("Final/datasets.xlsx")
datasets_path.parent.mkdir(parents=True, exist_ok=True)

datasets_df = (
    pd.read_parquet("Final/merged_all_final.parquet", columns=["datasetTitle", "datasetKey", "sourceDatabase"])
    .dropna()
    .groupby(["datasetTitle", "datasetKey", "sourceDatabase"], as_index=False)
    .size()
    .rename(columns={"size": "numberOfRecords"})
    .sort_values(["datasetTitle", "sourceDatabase"])
)

datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(datasets_df.shape)
print(datasets_df.head())

Saved: Final\datasets.xlsx
(437, 4)
                                        datasetTitle  \
0  1776-1997 Paul F. Clark North East Atlantic Cr...   
1  A Biotic Database of Indo-Pacific Marine Mollusks   
2  A new deepwater species of Calliopiidae, Halir...   
3  A new genus and five new species of the subfam...   
4  A revision of the genus Calcigorgia (Cnidaria,...   

                             datasetKey sourceDatabase  numberOfRecords  
0  67100d5f-8831-4a67-a553-fad388d96630           OBIS               54  
1  8bd1c7c0-717d-4a4e-8b1c-9ff3c5698067           OBIS               11  
2  9a23e0ce-ffdb-4ec1-9796-403ffb9ada90           GBIF                1  
3  56e22b1b-6708-4761-9a11-873c77c3b453           GBIF                5  
4  a9fc1eb5-487b-4f58-9c68-954498dd39a3           GBIF               15  


In [15]:
# Add new columns to datasets_df with string data type
new_columns = [
    "publisher",
    "publishingCountry",
    "abstract",
    "DOI",
    "citation",
    "license",
    "accessRights",
    "rightsHolder",
    "url",
    "published",
]

for col in new_columns:
    datasets_df[col] = pd.Series(dtype="string")

# Save to the same file
datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(datasets_df.head())

Saved: Final\datasets.xlsx
                                        datasetTitle  \
0  1776-1997 Paul F. Clark North East Atlantic Cr...   
1  A Biotic Database of Indo-Pacific Marine Mollusks   
2  A new deepwater species of Calliopiidae, Halir...   
3  A new genus and five new species of the subfam...   
4  A revision of the genus Calcigorgia (Cnidaria,...   

                             datasetKey sourceDatabase  numberOfRecords  \
0  67100d5f-8831-4a67-a553-fad388d96630           OBIS               54   
1  8bd1c7c0-717d-4a4e-8b1c-9ff3c5698067           OBIS               11   
2  9a23e0ce-ffdb-4ec1-9796-403ffb9ada90           GBIF                1   
3  56e22b1b-6708-4761-9a11-873c77c3b453           GBIF                5   
4  a9fc1eb5-487b-4f58-9c68-954498dd39a3           GBIF               15   

  publisher publishingCountry abstract   DOI citation license accessRights  \
0      <NA>              <NA>     <NA>  <NA>     <NA>    <NA>         <NA>   
1      <NA>              <NA>

## OGSL

In [16]:
# Add abstract text for OGSL entries
ogsl_abstract = """This dataset brings together data that was collected during the BIOME laboratory's annual scientific expeditions aboard the research icebreaker NGCC Amundsen, covering a period from 2007 to 2024. This long-term monitoring aims to document the structure of benthic communities and their habitats in the Canadian Arctic.

Endofauna was collected mainly using box corers (Box corer) and skips (Van Veen). The epifauna was collected mainly using an Agassiz trawl. The samples were sieved on a 500 µm mesh. For these two components, abundance and biomass metrics were standardized per unit area (m²) in order to facilitate inter-site and inter-annual comparisons.

In order to characterize the benthic habitat, physicochemical parameters were measured directly on the surface sediments taken by box corer. This data includes:

- Particle size, detailed in eight size classes (4000 µm, 2000 µm, 1000 µm, 500 µm, 250 µm, 125 µm, 63 µm, and less than 63 µm)
- Organic matter content
- The concentration of chlorophyll pigments (chlorophyll-a and pheopigments).

The data set includes a table of sampling events to accurately link biological and environmental data.

A complementary file, called "Contact List", also accompanies this data and documents the human and financial resources associated with expeditions. This file includes:
- The list of participants for each stage (leg) since 2007 ("Participants" tab)
- The history of funding for each donation since 2007 ("Financement" tab)
- The directory of all contacts ("Contacts" tab, grouping participants, donors, collaborators, etc.).

The section dedicated to funding illustrates the evolution and diversity of donors over the course of missions, highlighting in particular contributions to specific large-scale projects such as GreenEdge in 2016 or TCA in 2024. It should be noted that, although they are not explicitly listed for each individual legacy, the CRSNG and the FRQNT provided essential transversal financial support in the form of grants to the students who participated in the work.

It is important to note that this dataset is not exhaustive for the period from 2007 to 2024. It is the result of a vast effort of consolidation and standardization aimed at bringing together historical laboratory data. As a result, some missing data could be integrated as they are processed. In addition, this dataset is dynamic: it will be updated regularly to include data from future annual expeditions as soon as they are available."""

# Find OGSL entries in datasets_df
ogsl_rows = datasets_df["sourceDatabase"] == "OGSL"
datasets_df.loc[ogsl_rows, "abstract"] = ogsl_abstract

# Save to the same file
datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(f"Added abstract to {ogsl_rows.sum()} OGSL dataset(s)")

Saved: Final\datasets.xlsx
Added abstract to 1 OGSL dataset(s)


In [17]:
info_path = Path("OGSL/dataset_info.xlsx")

fill_cols = [
    "publisher",
    "publishingCountry",
    "DOI",
    "bibliographicCitation",
    "accessRights",
    "rightsHolder",
]

ogsl_info_df = pd.read_excel(info_path)

ogsl_info_df = (
    ogsl_info_df[["datasetKey", *fill_cols]]
    .dropna(subset=["datasetKey"])
    .drop_duplicates(subset=["datasetKey"])
    .set_index("datasetKey")
)

for col_name in fill_cols:
    if col_name in datasets_df.columns and col_name in ogsl_info_df.columns:
        datasets_df[col_name] = datasets_df[col_name].combine_first(
            datasets_df["datasetKey"].map(ogsl_info_df[col_name])
        )

datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")

Saved: Final\datasets.xlsx


## OBIS and GBIF

In [18]:
# Load the manual datasets file and map columns to datasets_df
manual_path = Path("Datasets/datasets_GBIF_OBIS.xlsx")
manual_df = pd.read_excel(manual_path)

# Create a mapping from alternateIdentifier to the manual data
manual_map = manual_df.set_index("alternateIdentifier").to_dict("index")

# Map the columns: abstract, intellectualRights, url, citation, pubDate
column_mapping = {
    "abstract": "abstract",
    "intellectualRights": "accessRights",
    "url": "url",
    "citation": "bibliographicCitation",
    "pubDate": "published",
    "licenseName": "license",
    "doi": "DOI",
    "publisher": "publisher",
}

# Update datasets_df using datasetKey as the lookup key
for idx, row in datasets_df.iterrows():
    dataset_key = row["datasetKey"]
    if dataset_key in manual_map:
        for manual_col, target_col in column_mapping.items():
            if manual_col in manual_df.columns and pd.notna(manual_map[dataset_key].get(manual_col)):
                datasets_df.at[idx, target_col] = manual_map[dataset_key][manual_col]

# Save to the same file
datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(f"Updated dataset metadata from OBIS/GBIF file")

Saved: Final\datasets.xlsx
Updated dataset metadata from OBIS/GBIF file


In [19]:
merged_path = Path("GBIF/merged_GBIF.parquet")
datasets_path = Path("Final/datasets.xlsx")

datasets_df = pd.read_excel(datasets_path)
gbif_df = pd.read_parquet(merged_path, columns=["datasetKey", "publisher", "publishingCountry", "rightsHolder"])

gbif_meta = (
    gbif_df.dropna(subset=["datasetKey"])
    .drop_duplicates(subset=["datasetKey"])
    .set_index("datasetKey")
)

for col in ["publisher", "publishingCountry", "rightsHolder"]:
    if col in datasets_df.columns and col in gbif_meta.columns:
        datasets_df[col] = datasets_df[col].combine_first(
            datasets_df["datasetKey"].map(gbif_meta[col])
        )
    elif col in gbif_meta.columns and col not in datasets_df.columns:
        datasets_df[col] = datasets_df["datasetKey"].map(gbif_meta[col])

datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")

Saved: Final\datasets.xlsx


In [20]:
merged_path = Path("OBIS/merged_OBIS.parquet")
datasets_path = Path("Final/datasets.xlsx")

datasets_df = pd.read_excel(datasets_path)
obis_df = pd.read_parquet(merged_path, columns=["dataset_id", "license", "rightsHolder"])

obis_meta = (
    obis_df.dropna(subset=["dataset_id"])
    .drop_duplicates(subset=["dataset_id"])
    .assign(dataset_id=lambda df: df["dataset_id"].astype(str))
    .set_index("dataset_id")
)

datasets_df["datasetKey"] = datasets_df["datasetKey"].astype(str)

for col in ["license", "rightsHolder"]:
    if col in datasets_df.columns and col in obis_meta.columns:
        datasets_df[col] = datasets_df[col].combine_first(
            datasets_df["datasetKey"].map(obis_meta[col])
        )
    elif col in obis_meta.columns and col not in datasets_df.columns:
        datasets_df[col] = datasets_df["datasetKey"].map(obis_meta[col])

datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")

Saved: Final\datasets.xlsx


## CRITTERBASE

In [21]:
cb_path = Path("CB/original.xlsx")
datasets_path = Path("Final/datasets.xlsx")

cb_df = pd.read_excel(cb_path, sheet_name="Dataset")

# Map CB "Name" -> final "datasetKey"
cb_df["Name"] = cb_df["Name"].astype("string")
datasets_df["datasetKey"] = datasets_df["datasetKey"].astype("string")

cb_map = (
    cb_df.dropna(subset=["Name"])
    .drop_duplicates(subset=["Name"])
    .set_index("Name")
)

column_mapping = {
    "Institution": "publisher",
    "OwnerCountry": "publishingCountry",
    "Abstract": "abstract",
    "DOI": "DOI",
    "AccessLevelData": "accessRights",
}

for source_col, target_col in column_mapping.items():
    if source_col in cb_map.columns:
        if target_col not in datasets_df.columns:
            datasets_df[target_col] = pd.NA
        datasets_df[target_col] = datasets_df[target_col].combine_first(
            datasets_df["datasetKey"].map(cb_map[source_col])
        )

datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")

Saved: Final\datasets.xlsx


## Manually remove duplicated datasets

### Environmental Monitoring database (MOD) DNV in GBIF and OBIS

In [22]:
# Analyze coverage and coordinate consistency for two datasetKeys
keys = [
    "bed78790-cbec-44af-82af-fe78e9692287",
    "e2c4a839-89b5-484f-a63c-df89c2f9f974",
]
check_cols = ["areaSampled", "volumeSampled_m3", "biomass", "trawlingDistance_m", "gear", "individualCount"]
coord_pairs = [
    ("decimalLatitudeStart", "decimalLongitudeStart"),
    ("decimalLatitude", "decimalLongitude"),
]

df = pd.read_parquet(final_path)

results = {}
for key in keys:
    sel = df[df["datasetKey"] == key]
    total = len(sel)
    summary = {"total_records": total}
    for col in check_cols:
        if col in sel.columns:
            non_null = sel[col].notna().sum()
            summary[col] = {"non_null": int(non_null), "total": int(total), "pct": float(non_null / total * 100) if total else 0.0}
        else:
            summary[col] = None

    coords_info = {}
    for lat_col, lon_col in coord_pairs:
        if lat_col in sel.columns and lon_col in sel.columns:
            coords = sel[[lat_col, lon_col]].dropna()
            unique_pairs = coords.drop_duplicates()
            coords_info[f"{lat_col}|{lon_col}"] = {
                "non_null_pairs": int(len(coords)),
                "unique_pairs": int(len(unique_pairs)),
                "sample_unique": unique_pairs.head(10).to_records(index=False).tolist(),
            }
        else:
            coords_info[f"{lat_col}|{lon_col}"] = None

    summary["coords"] = coords_info
    results[key] = summary

# Print per-dataset summaries
for key, info in results.items():
    print(f"\n--- DatasetKey: {key} ---")
    print(f"Total records: {info['total_records']}")
    for col in check_cols:
        v = info[col]
        if v is None:
            print(f"{col}: (column missing)")
        else:
            print(f"{col}: {v['non_null']}/{v['total']} ({v['pct']:.1f}%)")
    for pair, pi in info["coords"].items():
        if pi is None:
            print(f"Coords ({pair}): (columns missing)")
        else:
            print(f"Coords ({pair}): non-null pairs={pi['non_null_pairs']}, unique_pairs={pi['unique_pairs']}")
            if pi["unique_pairs"] <= 10:
                print(" Sample unique coords:", pi["sample_unique"])

# Check coordinate overlap between the two datasets for available coordinate pairs
print("\n--- Coordinate overlap between datasets ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates()
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates()
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")

# Check whether rounding coordinates increases overlap
round_decimals = 4
print(f"\n--- Coordinate overlap rounded to {round_decimals} decimals ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords after rounding")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared rounded coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")


--- DatasetKey: bed78790-cbec-44af-82af-fe78e9692287 ---
Total records: 796754
areaSampled: 0/796754 (0.0%)
volumeSampled_m3: 0/796754 (0.0%)
biomass: 0/796754 (0.0%)
trawlingDistance_m: 0/796754 (0.0%)
gear: 796754/796754 (100.0%)
individualCount: 796754/796754 (100.0%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=796754, unique_pairs=2415
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- DatasetKey: e2c4a839-89b5-484f-a63c-df89c2f9f974 ---
Total records: 795297
areaSampled: 0/795297 (0.0%)
volumeSampled_m3: 0/795297 (0.0%)
biomass: 0/795297 (0.0%)
trawlingDistance_m: 0/795297 (0.0%)
gear: 0/795297 (0.0%)
individualCount: 795297/795297 (100.0%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=795297, unique_pairs=2424
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- Coordinate overlap between datasets ---
decimalLatitudeStart|decimalLongitudeStart: dataset1_unique=2415, dataset2_unique=2424, intersection=151
 

In [23]:
dataset_key_to_remove = "e2c4a839-89b5-484f-a63c-df89c2f9f974"

final_df = pd.read_parquet(final_path)
final_df = final_df[final_df["datasetKey"] != dataset_key_to_remove]
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(f"Remaining rows: {len(final_df)}")

datasets_df = pd.read_excel(datasets_path)
datasets_df = datasets_df[datasets_df["datasetKey"] != dataset_key_to_remove]
datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(f"Remaining datasets: {len(datasets_df)}")

Saved: Final\merged_all_final.parquet
Remaining rows: 2941110
Saved: Final\datasets.xlsx
Remaining datasets: 436


### imr_mareano_rpsledge

In [24]:
# Analyze coverage and coordinate consistency for two datasetKeys
keys = [
    "e7f1c38e-2a85-4c6e-80aa-d052766a916f",
    "152259dc-9c20-4c1a-9644-8e4b509d4f73",
]
check_cols = ["areaSampled", "volumeSampled_m3", "biomass", "trawlingDistance_m", "gear", "individualCount"]
coord_pairs = [
    ("decimalLatitudeStart", "decimalLongitudeStart"),
    ("decimalLatitude", "decimalLongitude"),
]

df = pd.read_parquet(final_path)

results = {}
for key in keys:
    sel = df[df["datasetKey"] == key]
    total = len(sel)
    summary = {"total_records": total}
    for col in check_cols:
        if col in sel.columns:
            non_null = sel[col].notna().sum()
            summary[col] = {"non_null": int(non_null), "total": int(total), "pct": float(non_null / total * 100) if total else 0.0}
        else:
            summary[col] = None

    coords_info = {}
    for lat_col, lon_col in coord_pairs:
        if lat_col in sel.columns and lon_col in sel.columns:
            coords = sel[[lat_col, lon_col]].dropna()
            unique_pairs = coords.drop_duplicates()
            coords_info[f"{lat_col}|{lon_col}"] = {
                "non_null_pairs": int(len(coords)),
                "unique_pairs": int(len(unique_pairs)),
                "sample_unique": unique_pairs.head(10).to_records(index=False).tolist(),
            }
        else:
            coords_info[f"{lat_col}|{lon_col}"] = None

    summary["coords"] = coords_info
    results[key] = summary

# Print per-dataset summaries
for key, info in results.items():
    print(f"\n--- DatasetKey: {key} ---")
    print(f"Total records: {info['total_records']}")
    for col in check_cols:
        v = info[col]
        if v is None:
            print(f"{col}: (column missing)")
        else:
            print(f"{col}: {v['non_null']}/{v['total']} ({v['pct']:.1f}%)")
    for pair, pi in info["coords"].items():
        if pi is None:
            print(f"Coords ({pair}): (columns missing)")
        else:
            print(f"Coords ({pair}): non-null pairs={pi['non_null_pairs']}, unique_pairs={pi['unique_pairs']}")
            if pi["unique_pairs"] <= 10:
                print(" Sample unique coords:", pi["sample_unique"])

# Check coordinate overlap between the two datasets for available coordinate pairs
print("\n--- Coordinate overlap between datasets ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates()
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates()
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")

# Check whether rounding coordinates increases overlap
round_decimals = 4
print(f"\n--- Coordinate overlap rounded to {round_decimals} decimals ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords after rounding")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared rounded coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")


--- DatasetKey: e7f1c38e-2a85-4c6e-80aa-d052766a916f ---
Total records: 19157
areaSampled: 19137/19157 (99.9%)
volumeSampled_m3: 0/19157 (0.0%)
biomass: 18963/19157 (99.0%)
trawlingDistance_m: 0/19157 (0.0%)
gear: 19157/19157 (100.0%)
individualCount: 18473/19157 (96.4%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=19157, unique_pairs=338
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- DatasetKey: 152259dc-9c20-4c1a-9644-8e4b509d4f73 ---
Total records: 16969
areaSampled: 0/16969 (0.0%)
volumeSampled_m3: 0/16969 (0.0%)
biomass: 16780/16969 (98.9%)
trawlingDistance_m: 0/16969 (0.0%)
gear: 0/16969 (0.0%)
individualCount: 9305/16969 (54.8%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=16969, unique_pairs=289
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- Coordinate overlap between datasets ---
decimalLatitudeStart|decimalLongitudeStart: dataset1_unique=338, dataset2_unique=289, intersection=289
 Example sha

In [25]:
dataset_key_to_remove = "152259dc-9c20-4c1a-9644-8e4b509d4f73"

final_df = pd.read_parquet(final_path)
final_df = final_df[final_df["datasetKey"] != dataset_key_to_remove]
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(f"Remaining rows: {len(final_df)}")

datasets_df = pd.read_excel(datasets_path)
datasets_df = datasets_df[datasets_df["datasetKey"] != dataset_key_to_remove]
datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(f"Remaining datasets: {len(datasets_df)}")

Saved: Final\merged_all_final.parquet
Remaining rows: 2924141
Saved: Final\datasets.xlsx
Remaining datasets: 435


### DFO Quebec Region MLI museum collection

In [26]:
# Analyze coverage and coordinate consistency for two datasetKeys
keys = [
    "93efc16a-7eb1-4792-bdec-de1ea69b8c65",
    "1febfcdd-3e1b-46db-b73d-1bbe81ce22a6",
]
check_cols = ["areaSampled", "volumeSampled_m3", "biomass", "trawlingDistance_m", "gear", "individualCount"]
coord_pairs = [
    ("decimalLatitudeStart", "decimalLongitudeStart"),
    ("decimalLatitude", "decimalLongitude"),
]

df = pd.read_parquet(final_path)

results = {}
for key in keys:
    sel = df[df["datasetKey"] == key]
    total = len(sel)
    summary = {"total_records": total}
    for col in check_cols:
        if col in sel.columns:
            non_null = sel[col].notna().sum()
            summary[col] = {"non_null": int(non_null), "total": int(total), "pct": float(non_null / total * 100) if total else 0.0}
        else:
            summary[col] = None

    coords_info = {}
    for lat_col, lon_col in coord_pairs:
        if lat_col in sel.columns and lon_col in sel.columns:
            coords = sel[[lat_col, lon_col]].dropna()
            unique_pairs = coords.drop_duplicates()
            coords_info[f"{lat_col}|{lon_col}"] = {
                "non_null_pairs": int(len(coords)),
                "unique_pairs": int(len(unique_pairs)),
                "sample_unique": unique_pairs.head(10).to_records(index=False).tolist(),
            }
        else:
            coords_info[f"{lat_col}|{lon_col}"] = None

    summary["coords"] = coords_info
    results[key] = summary

# Print per-dataset summaries
for key, info in results.items():
    print(f"\n--- DatasetKey: {key} ---")
    print(f"Total records: {info['total_records']}")
    for col in check_cols:
        v = info[col]
        if v is None:
            print(f"{col}: (column missing)")
        else:
            print(f"{col}: {v['non_null']}/{v['total']} ({v['pct']:.1f}%)")
    for pair, pi in info["coords"].items():
        if pi is None:
            print(f"Coords ({pair}): (columns missing)")
        else:
            print(f"Coords ({pair}): non-null pairs={pi['non_null_pairs']}, unique_pairs={pi['unique_pairs']}")
            if pi["unique_pairs"] <= 10:
                print(" Sample unique coords:", pi["sample_unique"])

# Check coordinate overlap between the two datasets for available coordinate pairs
print("\n--- Coordinate overlap between datasets ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates()
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates()
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")

# Check whether rounding coordinates increases overlap
round_decimals = 4
print(f"\n--- Coordinate overlap rounded to {round_decimals} decimals ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords after rounding")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared rounded coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")


--- DatasetKey: 93efc16a-7eb1-4792-bdec-de1ea69b8c65 ---
Total records: 2279
areaSampled: 0/2279 (0.0%)
volumeSampled_m3: 0/2279 (0.0%)
biomass: 0/2279 (0.0%)
trawlingDistance_m: 0/2279 (0.0%)
gear: 2279/2279 (100.0%)
individualCount: 1888/2279 (82.8%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=2279, unique_pairs=575
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- DatasetKey: 1febfcdd-3e1b-46db-b73d-1bbe81ce22a6 ---
Total records: 2272
areaSampled: 0/2272 (0.0%)
volumeSampled_m3: 0/2272 (0.0%)
biomass: 0/2272 (0.0%)
trawlingDistance_m: 0/2272 (0.0%)
gear: 0/2272 (0.0%)
individualCount: 1875/2272 (82.5%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=2272, unique_pairs=572
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- Coordinate overlap between datasets ---
decimalLatitudeStart|decimalLongitudeStart: dataset1_unique=575, dataset2_unique=572, intersection=153
 Example shared coords: [(np.float64(69.52), 

In [27]:
dataset_key_to_remove = "1febfcdd-3e1b-46db-b73d-1bbe81ce22a6"

final_df = pd.read_parquet(final_path)
final_df = final_df[final_df["datasetKey"] != dataset_key_to_remove]
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(f"Remaining rows: {len(final_df)}")

datasets_df = pd.read_excel(datasets_path)
datasets_df = datasets_df[datasets_df["datasetKey"] != dataset_key_to_remove]
datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(f"Remaining datasets: {len(datasets_df)}")

Saved: Final\merged_all_final.parquet
Remaining rows: 2921869
Saved: Final\datasets.xlsx
Remaining datasets: 434


### imr_mareano_beamtrawl

In [28]:
# Analyze coverage and coordinate consistency for two datasetKeys
keys = [
    "99006141-bf02-4821-8e6e-5818a1cbf70a",
    "14fa3c3e-259c-4af9-9314-eee1dc3a119b",
]
check_cols = ["areaSampled", "volumeSampled_m3", "biomass", "trawlingDistance_m", "gear", "individualCount"]
coord_pairs = [
    ("decimalLatitudeStart", "decimalLongitudeStart"),
    ("decimalLatitude", "decimalLongitude"),
]

df = pd.read_parquet(final_path)

results = {}
for key in keys:
    sel = df[df["datasetKey"] == key]
    total = len(sel)
    summary = {"total_records": total}
    for col in check_cols:
        if col in sel.columns:
            non_null = sel[col].notna().sum()
            summary[col] = {"non_null": int(non_null), "total": int(total), "pct": float(non_null / total * 100) if total else 0.0}
        else:
            summary[col] = None

    coords_info = {}
    for lat_col, lon_col in coord_pairs:
        if lat_col in sel.columns and lon_col in sel.columns:
            coords = sel[[lat_col, lon_col]].dropna()
            unique_pairs = coords.drop_duplicates()
            coords_info[f"{lat_col}|{lon_col}"] = {
                "non_null_pairs": int(len(coords)),
                "unique_pairs": int(len(unique_pairs)),
                "sample_unique": unique_pairs.head(10).to_records(index=False).tolist(),
            }
        else:
            coords_info[f"{lat_col}|{lon_col}"] = None

    summary["coords"] = coords_info
    results[key] = summary

# Print per-dataset summaries
for key, info in results.items():
    print(f"\n--- DatasetKey: {key} ---")
    print(f"Total records: {info['total_records']}")
    for col in check_cols:
        v = info[col]
        if v is None:
            print(f"{col}: (column missing)")
        else:
            print(f"{col}: {v['non_null']}/{v['total']} ({v['pct']:.1f}%)")
    for pair, pi in info["coords"].items():
        if pi is None:
            print(f"Coords ({pair}): (columns missing)")
        else:
            print(f"Coords ({pair}): non-null pairs={pi['non_null_pairs']}, unique_pairs={pi['unique_pairs']}")
            if pi["unique_pairs"] <= 10:
                print(" Sample unique coords:", pi["sample_unique"])

# Check coordinate overlap between the two datasets for available coordinate pairs
print("\n--- Coordinate overlap between datasets ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates()
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates()
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")

# Check whether rounding coordinates increases overlap
round_decimals = 4
print(f"\n--- Coordinate overlap rounded to {round_decimals} decimals ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords after rounding")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared rounded coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")


--- DatasetKey: 99006141-bf02-4821-8e6e-5818a1cbf70a ---
Total records: 30984
areaSampled: 30967/30984 (99.9%)
volumeSampled_m3: 0/30984 (0.0%)
biomass: 30819/30984 (99.5%)
trawlingDistance_m: 0/30984 (0.0%)
gear: 30984/30984 (100.0%)
individualCount: 28607/30984 (92.3%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=30984, unique_pairs=411
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- DatasetKey: 14fa3c3e-259c-4af9-9314-eee1dc3a119b ---
Total records: 27426
areaSampled: 0/27426 (0.0%)
volumeSampled_m3: 0/27426 (0.0%)
biomass: 27265/27426 (99.4%)
trawlingDistance_m: 0/27426 (0.0%)
gear: 0/27426 (0.0%)
individualCount: 11241/27426 (41.0%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=27426, unique_pairs=354
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- Coordinate overlap between datasets ---
decimalLatitudeStart|decimalLongitudeStart: dataset1_unique=411, dataset2_unique=354, intersection=354
 Example sh

In [29]:
dataset_key_to_remove = "14fa3c3e-259c-4af9-9314-eee1dc3a119b"

final_df = pd.read_parquet(final_path)
final_df = final_df[final_df["datasetKey"] != dataset_key_to_remove]
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(f"Remaining rows: {len(final_df)}")

datasets_df = pd.read_excel(datasets_path)
datasets_df = datasets_df[datasets_df["datasetKey"] != dataset_key_to_remove]
datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(f"Remaining datasets: {len(datasets_df)}")

Saved: Final\merged_all_final.parquet
Remaining rows: 2894443
Saved: Final\datasets.xlsx
Remaining datasets: 433


### imr_mareano_grab

In [30]:
# Analyze coverage and coordinate consistency for two datasetKeys
keys = [
    "d30a617e-cac3-470c-a070-601376f45d6b",
    "d556b9d4-7625-4aa2-894d-441eabae47f7",
]
check_cols = ["areaSampled", "volumeSampled_m3", "biomass", "trawlingDistance_m", "gear", "individualCount"]
coord_pairs = [
    ("decimalLatitudeStart", "decimalLongitudeStart"),
    ("decimalLatitude", "decimalLongitude"),
]

df = pd.read_parquet(final_path)

results = {}
for key in keys:
    sel = df[df["datasetKey"] == key]
    total = len(sel)
    summary = {"total_records": total}
    for col in check_cols:
        if col in sel.columns:
            non_null = sel[col].notna().sum()
            summary[col] = {"non_null": int(non_null), "total": int(total), "pct": float(non_null / total * 100) if total else 0.0}
        else:
            summary[col] = None

    coords_info = {}
    for lat_col, lon_col in coord_pairs:
        if lat_col in sel.columns and lon_col in sel.columns:
            coords = sel[[lat_col, lon_col]].dropna()
            unique_pairs = coords.drop_duplicates()
            coords_info[f"{lat_col}|{lon_col}"] = {
                "non_null_pairs": int(len(coords)),
                "unique_pairs": int(len(unique_pairs)),
                "sample_unique": unique_pairs.head(10).to_records(index=False).tolist(),
            }
        else:
            coords_info[f"{lat_col}|{lon_col}"] = None

    summary["coords"] = coords_info
    results[key] = summary

# Print per-dataset summaries
for key, info in results.items():
    print(f"\n--- DatasetKey: {key} ---")
    print(f"Total records: {info['total_records']}")
    for col in check_cols:
        v = info[col]
        if v is None:
            print(f"{col}: (column missing)")
        else:
            print(f"{col}: {v['non_null']}/{v['total']} ({v['pct']:.1f}%)")
    for pair, pi in info["coords"].items():
        if pi is None:
            print(f"Coords ({pair}): (columns missing)")
        else:
            print(f"Coords ({pair}): non-null pairs={pi['non_null_pairs']}, unique_pairs={pi['unique_pairs']}")
            if pi["unique_pairs"] <= 10:
                print(" Sample unique coords:", pi["sample_unique"])

# Check coordinate overlap between the two datasets for available coordinate pairs
print("\n--- Coordinate overlap between datasets ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates()
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates()
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")

# Check whether rounding coordinates increases overlap
round_decimals = 4
print(f"\n--- Coordinate overlap rounded to {round_decimals} decimals ---")
for lat_col, lon_col in coord_pairs:
    if all(c in df.columns for c in (lat_col, lon_col)):
        a = df[df["datasetKey"] == keys[0]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        b = df[df["datasetKey"] == keys[1]][[lat_col, lon_col]].dropna().drop_duplicates().round(round_decimals)
        if a.empty or b.empty:
            print(f"{lat_col}|{lon_col}: one or both datasets have no non-null coords after rounding")
            continue
        set_a = set(map(tuple, a.to_records(index=False)))
        set_b = set(map(tuple, b.to_records(index=False)))
        inter = set_a & set_b
        print(f"{lat_col}|{lon_col}: dataset1_unique={len(set_a)}, dataset2_unique={len(set_b)}, intersection={len(inter)}")
        if len(inter) > 0:
            print(" Example shared rounded coords:", list(inter)[:10])
    else:
        print(f"{lat_col}|{lon_col}: coordinate columns not present in dataframe")


--- DatasetKey: d30a617e-cac3-470c-a070-601376f45d6b ---
Total records: 61382
areaSampled: 61349/61382 (99.9%)
volumeSampled_m3: 0/61382 (0.0%)
biomass: 60327/61382 (98.3%)
trawlingDistance_m: 0/61382 (0.0%)
gear: 61382/61382 (100.0%)
individualCount: 60667/61382 (98.8%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=61382, unique_pairs=979
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- DatasetKey: d556b9d4-7625-4aa2-894d-441eabae47f7 ---
Total records: 13710
areaSampled: 0/13710 (0.0%)
volumeSampled_m3: 0/13710 (0.0%)
biomass: 13668/13710 (99.7%)
trawlingDistance_m: 0/13710 (0.0%)
gear: 0/13710 (0.0%)
individualCount: 12868/13710 (93.9%)
Coords (decimalLatitudeStart|decimalLongitudeStart): non-null pairs=13710, unique_pairs=257
Coords (decimalLatitude|decimalLongitude): (columns missing)

--- Coordinate overlap between datasets ---
decimalLatitudeStart|decimalLongitudeStart: dataset1_unique=979, dataset2_unique=257, intersection=257
 Example sh

In [31]:
dataset_key_to_remove = "d556b9d4-7625-4aa2-894d-441eabae47f7"

final_df = pd.read_parquet(final_path)
final_df = final_df[final_df["datasetKey"] != dataset_key_to_remove]
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(f"Remaining rows: {len(final_df)}")

datasets_df = pd.read_excel(datasets_path)
datasets_df = datasets_df[datasets_df["datasetKey"] != dataset_key_to_remove]
datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(f"Remaining datasets: {len(datasets_df)}")

Saved: Final\merged_all_final.parquet
Remaining rows: 2880733
Saved: Final\datasets.xlsx
Remaining datasets: 432
